# pandas

pandas는 Python의 **표 형태 데이터(정형 데이터) 처리 라이브러리**입니다.  
데이터 분석, 전처리, 머신러닝 입력 준비 등 거의 모든 작업의 출발점이 됩니다.

## 1. 왜 필요한가?

### Python 기본 자료구조의 한계

```python
# 순수 Python으로 표 데이터를 다루면?
data = [
    ["Alice", 25, 88.5],
    ["Bob",   30, 72.0],
]
# 열 이름 없음 → data[0][2] 가 뭔지 바로 알 수 없음
# 결측값 처리, 필터링, 그룹 집계 → 모두 직접 구현해야 함
```

pandas는 이 문제를 해결합니다:

| 문제 | pandas 해결책 |
|------|---------------|
| 열 이름 없음 | `df["점수"]` — 이름으로 접근 |
| 결측값(NaN) 처리 곤란 | `fillna()`, `dropna()` |
| 느린 반복문 | NumPy 벡터 연산 기반 고속 처리 |
| 파일 I/O 번거로움 | `read_csv()`, `read_excel()`, `to_csv()` |
| 그룹 집계 | `groupby()` |

> 업계 표준: Kaggle, 논문, 기업 데이터 파이프라인의 대부분이 pandas로 시작합니다.

## 2. 어떤 걸 할 수 있는가?

```
파일 읽기/쓰기   →  탐색(EDA)  →  전처리  →  분석/시각화  →  ML 입력 준비
read_csv()         head()         fillna()    describe()       .values
read_excel()       info()         dropna()    groupby()        to_numpy()
read_json()        shape          astype()    merge()
                   dtypes         rename()    concat()
```

### 핵심 두 가지 자료구조

| 구조 | 설명 | 비유 |
|------|------|------|
| **Series** | 1차원 배열 + 인덱스 | 엑셀의 한 열 |
| **DataFrame** | 2차원 표 (Series들의 묶음) | 엑셀 시트 전체 |

## 3. 실제로 해보기

### 3-1. 데이터 만들기

In [ ]:
import pandas as pd
import numpy as np

# 방법 1: 딕셔너리 → DataFrame
df = pd.DataFrame({
    "name":   ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "age":    [25, 30, 22, None, 28],
    "score":  [88.5, 72.0, 95.3, 60.0, 81.0],
    "passed": [True, False, True, False, True],
    "dept":   ["Math", "CS", "Math", "CS", "Physics"],
})

print("shape:", df.shape)   # (행, 열)
print()
print(df)

In [ ]:
# 방법 2: 리스트 of 리스트 + 열 이름
df2 = pd.DataFrame(
    [["Alice", 25, 88.5],
     ["Bob",   30, 72.0]],
    columns=["name", "age", "score"]
)

# 방법 3: CSV 파일 읽기 (예시)
# df3 = pd.read_csv("data.csv")

print(df2)

### 3-2. Series — 1차원 열 하나

In [ ]:
# DataFrame의 한 열을 꺼내면 Series
scores = df["score"]
print(type(scores))   # <class 'pandas.core.series.Series'>
print(scores)
print()

# Series 직접 생성
s = pd.Series([10, 20, 30], index=["a", "b", "c"], name="example")
print(s)
print()

# Series 연산은 벡터 단위
print("평균:", scores.mean())
print("최대:", scores.max())
print("80점 이상:", (scores >= 80).sum(), "명")

### 3-3. DataFrame — 핵심 조작

In [ ]:
# --- 열 선택 ---
print(df[["name", "score"]])    # 여러 열 선택
print()

# --- 행 필터링 ---
print("합격자만:")
print(df[df["passed"] == True])
print()

# --- 열 추가 ---
df["grade"] = df["score"].apply(lambda s: "A" if s >= 90 else ("B" if s >= 80 else "C"))
print(df)

In [ ]:
# --- 인덱싱: loc (레이블) vs iloc (정수 위치) ---
print("loc[0] — 0번 행:")      # 레이블 기준
print(df.loc[0, ["name", "score"]])
print()

print("iloc[1:3] — 1~2번 행:")  # 위치 기준 (슬라이싱)
print(df.iloc[1:3])

### 3-4. info / describe / 결측값 처리

In [ ]:
# info(): 전체 구조 요약 — 열 이름, 타입, 결측값 유무
df.info()

In [ ]:
# describe(): 수치형 열의 기초 통계
df.describe().round(2)

In [ ]:
# 결측값 확인 및 처리
print("결측값 현황:")
print(df.isnull().sum())
print()

# fillna: 결측값을 평균으로 채움
df["age"] = df["age"].fillna(df["age"].mean())

# dropna: 결측값 있는 행 제거 (원본 유지하려면 inplace=False)
# df_clean = df.dropna()

print("처리 후 결측값:")
print(df.isnull().sum())
print()
print(df)

In [ ]:
# groupby: 그룹별 집계
print("학과별 평균 점수:")
print(df.groupby("dept")["score"].mean().round(2))
print()

print("학과별 합격자 수:")
print(df.groupby("dept")["passed"].sum())

### 3-5. merge와 concat

| 함수 | 용도 | SQL 비유 |
|------|------|----------|
| `pd.merge()` | 공통 키로 두 표를 합침 | JOIN |
| `pd.concat()` | 표를 위아래/좌우로 이어 붙임 | UNION / 열 추가 |

In [ ]:
# --- merge: 공통 키(name)로 두 DataFrame 합치기 ---
scores_df = pd.DataFrame({
    "name":  ["Alice", "Bob", "Carol"],
    "score": [88.5, 72.0, 95.3],
})

info_df = pd.DataFrame({
    "name": ["Alice", "Bob", "Dave"],
    "dept": ["Math", "CS", "CS"],
})

# inner: 양쪽 모두 있는 행만 (교집합)
inner = pd.merge(scores_df, info_df, on="name", how="inner")
print("inner join:")
print(inner)
print()

# left: 왼쪽 DataFrame 기준 (왼쪽 행은 모두 유지)
left = pd.merge(scores_df, info_df, on="name", how="left")
print("left join:")
print(left)

In [ ]:
# --- concat: 행 방향으로 이어 붙이기 (axis=0, 기본값) ---
batch1 = pd.DataFrame({"name": ["Alice", "Bob"],   "score": [88.5, 72.0]})
batch2 = pd.DataFrame({"name": ["Carol", "Dave"],  "score": [95.3, 60.0]})

combined = pd.concat([batch1, batch2], ignore_index=True)
print("행 concat:")
print(combined)
print()

# --- concat: 열 방향으로 이어 붙이기 (axis=1) ---
names_s  = pd.Series(["Alice", "Bob", "Carol"], name="name")
scores_s = pd.Series([88.5,  72.0,   95.3],   name="score")
grades_s = pd.Series(["A",   "C",    "A"],     name="grade")

side_by_side = pd.concat([names_s, scores_s, grades_s], axis=1)
print("열 concat:")
print(side_by_side)

In [ ]:
# merge vs concat 비교 요약
print("""
merge  — 공통 키로 두 표를 옆으로 연결 (SQL JOIN)
         pd.merge(df_a, df_b, on="key", how="inner/left/right/outer")

concat — 표를 그냥 이어 붙이기
         axis=0 → 행 추가 (새 데이터 배치 합치기)
         axis=1 → 열 추가 (서로 다른 특성 붙이기)
""")

---

## 4. Polars — 대용량 데이터를 위한 차세대 라이브러리

pandas는 강력하지만, **수백만 행 이상**에서는 느려집니다.  
이때 대안이 **Polars**입니다.

### pandas vs Polars 비교

| 항목 | pandas | Polars |
|------|--------|--------|
| 언어 기반 | Python / NumPy | Rust (초고속) |
| 멀티스레드 | X (기본 단일) | O (자동 병렬) |
| 메모리 효율 | 보통 | 낮은 메모리 사용 |
| 문법 | 익숙한 Python | 다소 다름, 일관적 |
| 대용량 성능 | 느려짐 | 수십 배 빠름 |
| 생태계 | 매우 성숙 | 빠르게 성장 중 |

> **결론**: 탐색·전처리 단계는 pandas, 수백만 행 이상의 파이프라인은 Polars를 고려하세요.

In [ ]:
# Polars 설치 (처음 한 번만)
# !pip install polars

import polars as pl

# pandas와 거의 같은 방식으로 DataFrame 생성
df_pl = pl.DataFrame({
    "name":   ["Alice", "Bob", "Carol", "Dave"],
    "age":    [25, 30, 22, 28],
    "score":  [88.5, 72.0, 95.3, 60.0],
    "passed": [True, False, True, False],
})

print(df_pl)

In [ ]:
# Polars의 표현식(Expression) 문법 — pandas와의 차이

# 필터링
print("합격자:")
print(df_pl.filter(pl.col("passed") == True))
print()

# 열 추가
df_pl = df_pl.with_columns(
    pl.when(pl.col("score") >= 90)
      .then(pl.lit("A"))
      .when(pl.col("score") >= 80)
      .then(pl.lit("B"))
      .otherwise(pl.lit("C"))
      .alias("grade")
)

# 그룹 집계
print("합격 여부별 평균 점수:")
print(
    df_pl.group_by("passed")
         .agg(pl.col("score").mean().alias("avg_score"))
         .sort("passed")
)

In [ ]:
# pandas ↔ Polars 변환
import pandas as pd

# Polars → pandas
df_pandas = df_pl.to_pandas()
print(type(df_pandas))

# pandas → Polars
df_back = pl.from_pandas(df_pandas)
print(type(df_back))

In [ ]:
# 성능 비교 (간단 벤치마크)
import time

N = 1_000_000
data = {"x": range(N), "y": range(N)}

# pandas
df_pd = pd.DataFrame(data)
t0 = time.perf_counter()
_ = df_pd.groupby("x")["y"].mean()
t_pd = time.perf_counter() - t0

# Polars
df_pl2 = pl.DataFrame({"x": list(range(N)), "y": list(range(N))})
t0 = time.perf_counter()
_ = df_pl2.group_by("x").agg(pl.col("y").mean())
t_pl = time.perf_counter() - t0

print(f"pandas  groupby 1M rows: {t_pd:.3f}s")
print(f"Polars  groupby 1M rows: {t_pl:.3f}s")
print(f"Polars가 {t_pd / t_pl:.1f}배 빠름")

---

## 정리

| 개념 | 핵심 한 줄 |
|------|------------|
| **Series** | 인덱스가 붙은 1차원 배열 |
| **DataFrame** | Series의 묶음, 엑셀 시트 |
| **info / describe** | 구조 확인 / 기초 통계 |
| **merge** | 공통 키로 두 표 합치기 (JOIN) |
| **concat** | 표를 위아래/좌우로 이어 붙이기 |
| **Polars** | 대용량에서 pandas보다 수십 배 빠른 대안 |

**다음 단계:**
- `dp_03_iris.ipynb` — 실제 데이터셋으로 전체 EDA 파이프라인
- `dp_04_matplotlib.ipynb` — 시각화 심화
- `dp_05_titanic.ipynb` — 결측값 처리 + 피처 엔지니어링